# UNet++ on the AI4Mars subset

UNet++ is a U-Net whose skip connections are replaced by dense, nested convolution blocks, so each decoder level gets refined features from every shallower level. We use `segmentation_models_pytorch` (`smp.UnetPlusPlus`) with an ImageNet-pretrained ResNet encoder.

Everything else (data, splits, loss, metrics, checkpointing) matches the SegFormer notebook so the results can be compared. Run all, top to bottom, on a T4 GPU.

1. Setup and settings
2. Load the data from the shared subset zip: train images and the test images (min2 gold masks plus extra big-rock images)
3. Train (best epoch chosen by test mIoU, the same way as the other models in the group)
4. Per-class IoU and confusion matrix of the best model on the test set

Training is skipped if a finished run with the same settings is already saved in Drive. Set RETRAIN = True to train again anyway.

In [ ]:
# Setup: install, imports, GPU check, Drive

!pip install -q segmentation-models-pytorch

import os, json, time, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from google.colab import drive

# Needs a GPU: Runtime > Change runtime type > T4 GPU
print("GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Switch the runtime to T4 GPU and run again."

drive.mount("/content/drive")


In [ ]:
# Settings. Change values here, not further down.

SEED = 42
SIZE = 512              # images and masks are resized to SIZE x SIZE (must be divisible by 32)
BATCH_SIZE = 4             # UNet++ at 512x512 runs out of memory on a T4 with 16
LR = 0.0001
EPOCHS = 20             # same as the SegFormer notebook
NUM_CLASSES = 4
CLASS_NAMES = ["soil", "bedrock", "sand", "big rock"]
IGNORE = 255            # unlabeled pixels, skipped in the loss and in IoU

ENCODER = "resnet34"    # try "resnet50" for a bigger encoder

RETRAIN = False         # True = train again even if a finished run is saved

# Data
SUBSET_ZIP = "/content/drive/MyDrive/AI4Mars_Subset/ai4mars_subset.zip"
SUBSET_ROOT = "/content/ai4mars_subset"

# Outputs, saved in Drive so they survive a runtime reset
RUN_DIR = "/content/drive/MyDrive/unetpp_runs"
os.makedirs(RUN_DIR, exist_ok=True)

device = "cuda"


In [ ]:
# Get the data on disk.
# If a different version of the zip is already unzipped in this runtime, it is replaced.

with zipfile.ZipFile(SUBSET_ZIP) as z:
    zip_files = sum(not n.endswith("/") for n in z.namelist())
    disk_files = sum(len(fs) for _, _, fs in os.walk(SUBSET_ROOT)) if os.path.exists(SUBSET_ROOT) else -1
    if disk_files != zip_files:
        shutil.rmtree(SUBSET_ROOT, ignore_errors=True)
        z.extractall(SUBSET_ROOT)
        print("unzipped subset")

IMG_DIR = f"{SUBSET_ROOT}/images/edr"
TRAIN_MASKS = f"{SUBSET_ROOT}/labels/train"
TEST_MASKS = f"{SUBSET_ROOT}/labels/test"

def png_names(folder):
    """Mask file names in a folder, without .png"""
    return {f[:-4] for f in os.listdir(folder) if f.endswith(".png")}

print("train masks:", len(png_names(TRAIN_MASKS)))
print("test masks: ", len(png_names(TEST_MASKS)))


In [ ]:
# Pair each mask with its photo, and build the train / test lists

def pair_files(mask_dir):
    """Return a sorted list of (photo path, mask path).
    Min2 test masks end in _merged and their photos do not."""
    pairs, missing = [], []
    for name in sorted(png_names(mask_dir)):
        img_path = os.path.join(IMG_DIR, name.removesuffix("_merged") + ".JPG")
        if os.path.exists(img_path):
            pairs.append((img_path, os.path.join(mask_dir, name + ".png")))
        else:
            missing.append(name)
    if missing:
        print(f"skipped {len(missing)} masks with no photo in {mask_dir}, e.g. {missing[:2]}")
    return pairs

train_pairs = pair_files(TRAIN_MASKS)
test_pairs = pair_files(TEST_MASKS)

# Same as the other models in the group: the test set also picks the best epoch
# (there is no separate val set), so every model is compared on the same images.
val_pairs = test_pairs

# Two kinds of test images: min2 gold labels (_merged) and the extra big-rock images
n_gold = sum(m.endswith("_merged.png") for _, m in test_pairs)

print("train:", len(train_pairs))
print("test: ", len(test_pairs), f"({n_gold} min2 + {len(test_pairs) - n_gold} extra big rock)")


In [ ]:
# Dataset and loaders

# ImageNet averages. The encoder was pretrained on images normalized this way.
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

class MarsDataset(Dataset):
    """Loads one photo and its mask, both resized to SIZE x SIZE."""
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        img_path, mask_path = self.pairs[i]
        # Photo: grayscale to 3 channels (the pretrained encoder expects 3), smooth resize
        img = Image.open(img_path).convert("RGB").resize((SIZE, SIZE), Image.BILINEAR)
        # Mask: nearest resize so class numbers stay exact
        mask = Image.open(mask_path).resize((SIZE, SIZE), Image.NEAREST)

        img = (np.array(img, dtype=np.float32) / 255.0 - MEAN) / STD
        img = torch.from_numpy(img).permute(2, 0, 1)            # (H, W, C) -> (C, H, W)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))  # class IDs as whole numbers
        return img, mask

def make_loader(pairs, shuffle):
    return DataLoader(MarsDataset(pairs), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2)

val_loader = make_loader(val_pairs, shuffle=False)
test_loader = make_loader(test_pairs, shuffle=False)

# Check one batch
imgs, masks = next(iter(val_loader))
print("images:", imgs.shape, imgs.dtype)
print("masks: ", masks.shape, masks.dtype)
print("mask values:", torch.unique(masks).tolist())   # should only be 0, 1, 2, 3, 255

In [ ]:
# Model, metrics, and training

torch.backends.cudnn.benchmark = True   # faster convolutions when input sizes never change

def build_model(pretrained=True):
    torch.manual_seed(SEED)
    return smp.UnetPlusPlus(
        encoder_name=ENCODER,
        encoder_weights="imagenet" if pretrained else None,
        in_channels=3,
        classes=NUM_CLASSES,     # raw logits out; CrossEntropyLoss applies softmax
    ).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE)

def count_pixels(preds, masks):
    """Confusion matrix counts for one batch: row = true class, column = predicted class.
    Unlabeled pixels are skipped."""
    valid = masks != IGNORE
    idx = masks[valid] * NUM_CLASSES + preds[valid]
    return torch.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def iou_per_class(cm):
    """IoU for each class from a confusion matrix (numpy). nan if a class never appears."""
    inter = np.diag(cm).astype(float)
    union = cm.sum(axis=0) + cm.sum(axis=1) - inter
    with np.errstate(invalid="ignore", divide="ignore"):
        return inter / union

def mean_iou(cm):
    """Average IoU over the classes that appear."""
    return float(np.nanmean(iou_per_class(cm)))

def run_epoch(model, loader, optimizer=None, scaler=None):
    """One pass over loader. Trains if an optimizer (and its GradScaler) is given, otherwise only evaluates.
    Uses mixed precision (float16) on the GPU to save memory and time.
    Returns (average loss, mIoU, confusion matrix)."""
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long, device=device)

    start = time.time()
    for step, (imgs, masks) in enumerate(loader, 1):
        imgs, masks = imgs.to(device), masks.to(device)
        with torch.set_grad_enabled(training):
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(imgs)
            loss = loss_fn(logits.float(), masks)   # CrossEntropyLoss applies softmax internally
        if training:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        if step in (1, 5, 20) or step % 50 == 0:   # progress, so a slow epoch is not a silent one
            print(f"  {'train' if training else 'test'} step {step}/{len(loader)} | {time.time() - start:.0f}s", flush=True)

        total_loss += loss.item()
        preds = logits.softmax(dim=1).argmax(dim=1)   # most likely class per pixel
        cm += count_pixels(preds, masks)

    cm = cm.cpu().numpy()
    return total_loss / len(loader), mean_iou(cm), cm

def run_settings(train_pairs):
    """Settings that define a run. A saved run is reused only if these match."""
    return {"lr": LR, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "size": SIZE,
            "seed": SEED, "encoder": ENCODER, "amp": True, "n_train": len(train_pairs), "n_val": len(val_pairs)}

def train_run(name, train_pairs, save_dir):
    """Train a fresh model, keep the epoch with the best mIoU on the test set, and save everything to save_dir.
    Skips training if save_dir already has a finished run with the same settings."""
    weights_file = f"{save_dir}/best_unetpp.pt"
    history_file = f"{save_dir}/history.json"
    info_file = f"{save_dir}/run_info.json"   # written only when a run finishes
    settings = run_settings(train_pairs)

    if not RETRAIN and os.path.exists(info_file) and os.path.exists(weights_file):
        with open(info_file) as f:
            saved = json.load(f)
        if saved == settings:
            print(f"{name}: finished run with the same settings found in {save_dir}, skipping training")
            with open(history_file) as f:
                return json.load(f)
        print(f"{name}: saved run has different settings, training again")

    if os.path.exists(info_file):
        os.remove(info_file)   # so a half-finished run is never mistaken for a finished one

    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler("cuda")
    train_loader = make_loader(train_pairs, shuffle=True)
    history = {"train_loss": [], "val_loss": [], "train_miou": [], "val_miou": []}
    best = -1.0

    print(f"{name}: training on {len(train_pairs)} images for {EPOCHS} epochs")
    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        tr_loss, tr_miou, _ = run_epoch(model, train_loader, optimizer, scaler)
        va_loss, va_miou, _ = run_epoch(model, val_loader)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_miou"].append(tr_miou)
        history["val_miou"].append(va_miou)

        if va_miou > best:
            best = va_miou
            torch.save(model.state_dict(), weights_file)
        with open(history_file, "w") as f:
            json.dump(history, f)

        print(f"epoch {epoch} | train loss {tr_loss:.3f} miou {tr_miou:.3f} | "
              f"test loss {va_loss:.3f} miou {va_miou:.3f} | {time.time() - start:.0f}s")

    with open(info_file, "w") as f:
        json.dump(settings, f)
    best_epoch = int(np.argmax(history["val_miou"])) + 1
    print(f"{name}: best test mIoU {best:.3f} at epoch {best_epoch}")
    return history

def plot_run(name, history, n_train, save_dir):
    """Loss and mIoU vs epochs for one run, with the settings in red. Returns the file names."""
    epochs = range(1, len(history["train_loss"]) + 1)
    best_epoch = int(np.argmax(history["val_miou"])) + 1
    settings = (f"UNet++ {ENCODER} (ImageNet) | Adam lr={LR} | batch {BATCH_SIZE} | {SIZE}x{SIZE} | "
                f"{n_train} train images\n"
                f"CrossEntropy (ignore 255) | test: {len(val_pairs)} test images | {len(epochs)} epochs | "
                f"best test mIoU {max(history['val_miou']):.3f} @ epoch {best_epoch}")
    filenames = []
    for key, label in [("loss", "Loss"), ("miou", "mIoU")]:
        plt.figure(figsize=(9, 5.5))
        plt.plot(epochs, history[f"train_{key}"], marker="o", label="train")
        plt.plot(epochs, history[f"val_{key}"], marker="o", label="test")
        plt.xlabel("Epoch")
        plt.ylabel(label)
        plt.title(f"UNet++ {name}: {label} vs Epochs")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.figtext(0.5, 0.01, settings, ha="center", color="red", fontsize=9)
        plt.tight_layout(rect=[0, 0.08, 1, 1])
        filename = f"unetpp_{name}_{key}.png"
        plt.savefig(filename, dpi=150)
        plt.savefig(f"{save_dir}/{filename}", dpi=150)
        plt.show()
        filenames.append(filename)
    return filenames

print("ready")

# Train
Trains on the subset only.

In [ ]:
# Train UNet++ (check the time per epoch on a T4 and adjust SIZE / BATCH_SIZE if needed)

history = train_run("unetpp", train_pairs, RUN_DIR)
plots = plot_run("unetpp", history, len(train_pairs), RUN_DIR)

# Test
The best epoch (by test mIoU) is loaded again and scored on the test set.

In [ ]:
# Per-class IoU and confusion matrix on the test set

test_loader = make_loader(test_pairs, shuffle=False)
model = build_model(pretrained=False)
model.load_state_dict(torch.load(f"{RUN_DIR}/best_unetpp.pt", map_location=device))
model.eval()
with torch.no_grad():
    _, test_miou, test_cm = run_epoch(model, test_loader)

table = pd.DataFrame({"IoU": iou_per_class(test_cm)}, index=CLASS_NAMES)
table.loc["mIoU"] = test_miou
print(f"Test IoU ({len(test_pairs)} images)")
print(table.round(3).to_string())

# Confusion matrix as row percentages (each row sums to 1)
pct = test_cm / np.maximum(test_cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(pct, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES)
ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title(f"UNet++ (test mIoU {test_miou:.3f})")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{pct[i, j]:.2f}", ha="center", va="center",
                color="white" if pct[i, j] > 0.5 else "black")
fig.text(0.5, 0.01, f"UNet++ {ENCODER} (ImageNet) | Adam lr={LR} | batch {BATCH_SIZE} | {SIZE}x{SIZE} | "
                    f"{EPOCHS} epochs | test: {len(test_pairs)} images ({n_gold} min2 + {len(test_pairs) - n_gold} extra big rock)",
         ha="center", color="red", fontsize=8)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig("unetpp_confusion_matrix.png", dpi=150)
plt.savefig(f"{RUN_DIR}/unetpp_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# Download the graphs (the browser may ask to allow multiple downloads)

from google.colab import files

for f in plots + ["unetpp_confusion_matrix.png"]:
    files.download(f)